
# Supervisor Agent

Based on [this Databricks doc](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/multi-agent-supervisor)


**Supervisor Agent** is a multi-agent supervisor system that orchestrates AI agents and tools to work together.

Supervisor Agent is used to create a supervisor system that coordinates [Genie Spaces](https://docs.databricks.com/aws/en/genie/), AI agents (via agent endpoints), [Unity Catalog functions](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool), MCP tools and servers for complex workflows and deep analysis.

Supervisor Agent uses advanced AI orchestration patterns to manage agent interactions, task delegation, and result synthesis to deliver comprehensive solutions.

Supervisor Agent builds the system for you and lets you improve it over time with human feedback.

You can select up to 20 different agents and tools.


## Execution Permissions

When creating a supervisor, you must provide subagents for it to coordinate.

End users should be granted explicit access to each one.

The supervisor has built-in access controls, so that its end users only access the subagents and data they have access to.

If the end user does not have access to any subagents, the supervisor will end the conversation.
If the end user has access to some but not all subagents, the supervisor will redirect the conversation away from subagents the user cannot access.


## Default Storage

Supervisor Agent uses [default storage](https://docs.databricks.com/aws/en/storage/default-storage) to store temporary data transformations, model checkpoints, and internal metadata that power each agent.

On agent deletion, all data associated with the agent is removed from default storage.


## Workspace Requirements

* [Supported region](https://docs.databricks.com/aws/en/resources/feature-region-support#ai-agent-features-availability)
* [Serverless compute](https://docs.databricks.com/aws/en/compute/serverless/)
* [Serverless usage policy](https://docs.databricks.com/aws/en/admin/usage/budget-policies)
* [Unity Catalog](https://docs.databricks.com/aws/en/data-governance/unity-catalog/enable-workspaces)
* [Model Serving](https://docs.databricks.com/aws/en/machine-learning/model-serving/)
* [others...](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/multi-agent-supervisor#requirements)


## Supported Tools and Sub-agents

[Supported subagents and tools](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/multi-agent-supervisor#supported-subagents-and-tools)

* Genie spaces
* Knowledge Assistants
* UC Functions (incl. `python_exec`)
* External MCP
* Vector Search Indexes
* Databricks Apps
* Serving Endpoints
* Dashboards


## Description

For better results, provide a description for each tool/subagent.

The supervisor uses the information in the description to help it coordinate agents.

Provide as much detail as possible to help improve its task delegation.


## Testing

Chat with the agent to evaluate its responses.

Test the agent in [AI Playground](https://docs.databricks.com/aws/en/large-language-models/ai-playground).


# Demo: Create AI agent tools using Unity Catalog functions

[Create AI agent tools using Unity Catalog functions](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool)

Use Unity Catalog functions to create AI agent tools that execute custom logic and perform specific tasks that extend the capabilities of LLMs beyond language generation.

Unity Catalog tools are really just [user-defined functions (UDFs) in Unity Catalog](https://docs.databricks.com/aws/en/udf/unity-catalog) under the hood:

* `create_python_function` accepts a Python callable.
* `create_function` accepts a SQL body create function statement (incl. [Python functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function#create-python-functions)).

## Requirements

🚨 NOTE: [Requirements](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#requirements) are not clear what the requirements are for creating and using UC functions. There are three paragraphs.

* Databricks Runtime 15.0 or later
* Python 3.10 or later
* [Serverless compute](https://docs.databricks.com/aws/en/compute/serverless/#requirements) must be enabled to execute Unity Catalog functions as AI agent tools in production.

## Unity Catalog functions vs. MCP servers

[When to use Unity Catalog functions vs. MCP servers](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#when-to-use-unity-catalog-functions-vs-mcp-servers)

In most cases, Databricks recommends MCP servers or defining the logic directly in agent code.

Databricks recommends using MCP servers to add Unity Catalog functions to your agent.

Use Unity Catalog functions as agent tools specifically for structured data retrieval tools when the query is known ahead of time and the agent provides the parameters.

## Create Unity Catalog Function

In [0]:
# Install Unity Catalog AI integration packages with the Databricks extra
# Installs unitycatalog-ai-0.4.0 unitycatalog-client-0.4.1
%pip install -U unitycatalog-ai[databricks]>=0.4.0 mlflow-skinny[databricks]>=3.12.0 databricks-agents>=1.10.2 databricks-mcp>=0.9.0
%restart_python

In [0]:
%pip show unitycatalog-ai

In [0]:
%pip show databricks-mcp

In [0]:
%pip show mlflow-skinny

In [0]:
%pip show databricks-agents


[Databricks Function Client](https://docs.unitycatalog.io/ai/client/#databricks-function-client)

`DatabricksFunctionClient` is a core component of the **Unity Catalog AI Core Library** to interact with Unity Catalog (UC) functions on Databricks.

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

In [0]:
def add_numbers(number_1: float, number_2: float) -> float:
  """
  A function that accepts two floating point numbers adds them,
  and returns the resulting sum as a float.

  Args:
    number_1 (float): The first of the two numbers to add.
    number_2 (float): The second of the two numbers to add.

  Returns:
    float: The sum of the two input numbers.
  """
  return number_1 + number_2

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS jacek_agents;
CREATE SCHEMA IF NOT EXISTS jacek_agents.default;

In [0]:
CATALOG = "jacek_agents"
SCHEMA = "default"

function_info = client.create_python_function(
  func=add_numbers,
  catalog=CATALOG,
  schema=SCHEMA,
  replace=True
)

In [0]:
result = client.execute_function(
  function_name=f"{CATALOG}.{SCHEMA}.add_numbers",
  parameters={"number_1": 36939.0, "number_2": 8922.4}
)

# client.execute_function returns a string (not a float)
assert float(result.value) == 45861.4

## Add Unity Catalog function to Agent using MCP

[Add Unity Catalog functions to your agent](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#add-unity-catalog-functions-to-your-agent):

There are two approaches to add UC functions to agents:

* Using MCP (recommended)
* Using `UCFunctionToolkit` (to integrate with LangChain, LlamaIndex, OpenAI, Anthropic, and [other third party generative AI frameworks](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unity-catalog-tool-integration))

The recommended MCP approach provides a simpler integration with automatic tool discovery and built-in authentication support.

### Managed MCP URL for Unity Catalog functions

The managed MCP URL for Unity Catalog functions is: `https://<workspace-hostname>/api/2.0/mcp/functions/{catalog}/{schema}`.

You can optionally specify a specific function by appending `/{function_name}`.

### Deploy Agent on Model Serving for GenAI Apps

[Databricks recommends deploying agents on Databricks Apps for full control over agent code, server configuration, and deployment workflow](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent)

Before deploying an agent, you have to register the agent to Unity Catalog.

Registering the agent packages it as a model in Unity Catalog.
As a result, you can use Unity Catalog permissions for authorization for resources in the agent.

In [0]:
# Databricks notebooks already run inside an active asyncio event loop
# DatabricksMCPClient.list_tools() internally calls asyncio.run()
# and fails with RuntimeError: asyncio.run() cannot be called from a running event loop
import nest_asyncio
nest_asyncio.apply()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
import mlflow

workspace_client = WorkspaceClient()
host = workspace_client.config.host

# Connect to the UC functions MCP server
mcp_client = DatabricksMCPClient(
    server_url=f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}",
    workspace_client=workspace_client,
)

# List available tools
tools = mcp_client.list_tools()
tools


# Demo: Create a multi-agent supervisor system

[Create a multi-agent supervisor system](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/multi-agent-supervisor#create-a-multi-agent-supervisor-system)


Agents > Create Agent > Supervisor Agent